In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_gold/08_dim_patient
# Purpose  : dim_patient with SCD Type 2 via dlt.apply_changes (stored_as_scd_type=2).
#            The project's sole SCD2 dimension, driven by patient payer transition history.
#
# Key Technical Notes:
#   1) Mandatory DLT system column names __START_AT and __END_AT are preserved
#      to prevent breaking native SCD2 mechanisms.
#   2) is_current is derived in a downstream View (__END_AT IS NULL).
#   3) Deterministic Surrogate Key generated via xxhash64(patient_id).
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.view(name="v_patient_changes")
def v_patient_changes():
    df = dlt.read_stream("covid19_lakehouse.silver.patients")
    return df.select(
        F.xxhash64("patient_id").alias("patient_key"),
        "patient_id",
        "first_name",
        "last_name",
        "birth_date",
        "death_date",
        "gender",
        "race",
        "ethnicity",
        "marital_status",
        "address",
        "city",
        "state",
        "county",
        "zip",
        "lat",
        "lon",
        "_ingested_at",
    )


dlt.create_streaming_table("covid19_lakehouse.gold.dim_patient_scd")

dlt.apply_changes(
    target="covid19_lakehouse.gold.dim_patient_scd",
    source="v_patient_changes",
    keys=["patient_id"],
    sequence_by=F.col("_ingested_at"),
    stored_as_scd_type=2,
    # Excludes these metadata columns from the "has the row actually changed?" comparison.
    # This prevents creating spurious SCD2 versions if an identical record is re-ingested
    # with a newer _ingested_at timestamp. This serves the exact same purpose as a manual
    # row_hash column, but leverages DLT's native implementation.
    track_history_except_column_list=["_ingested_at", "patient_key"],
)


# ============================================================
# Final View adding is_current (derived from __END_AT) to simplify daily queries
# (WHERE is_current = true) without needing to explicitly check __END_AT IS NULL each time.
# ============================================================
@dlt.table(
    name="covid19_lakehouse.gold.dim_patient",
    comment="Patient dimension - SCD Type 2, with is_current derived from __END_AT",
)
def dim_patient():
    df = spark.read.table("covid19_lakehouse.gold.dim_patient_scd")
    return df.withColumn("is_current", F.col("__END_AT").isNull())